# SOEN 400 – Customer Segmentation for a Retail Business using Clustering
## Phase 2: K-Means Clustering of Customer-Level RFM Data — Group 3

## 1. Project Overview

**Problem.** A UK-based online retailer serves thousands of customers who differ in how recently, how often and how much they buy. There are no predefined customer groups, so the groups must be *discovered* from purchasing behaviour. This is an **unsupervised learning** problem.

**What was already done (Phase 1, `Online_Retail_EDA_Jupyter.ipynb`).** Initial examination, data-quality analysis, cleaning decisions and construction of a customer-level **RFM** table (4,338 customers). This notebook **reuses that pipeline unchanged** and continues from it.

**What this notebook adds (Phase 2).**

| Stage | Sections |
|---|---|
| Reproduce the Phase 1 cleaning and RFM table (and verify it against the saved file) | 3 – 6 |
| Transform and standardize the RFM features | 7 – 8 |
| Illustrate how K-Means works on a small example | 9 |
| Select K with the elbow method, silhouette analysis and Calinski-Harabasz | 10 – 11 |
| Train the final K-Means model and assign customers to clusters | 12 – 13 |
| Profile, visualize, evaluate and interpret the clusters | 14 – 17 |
| Conclusions | 18 |

**Pipeline.** Transaction data → cleaning → customer-level RFM → log transform → standardization → K-Means → cluster profiles.

**How to run.** Open this notebook from the project folder (the folder that contains `Online Retail.csv`) and choose *Run All*. Every number, table and figure below is produced by the code; figures are saved to `figures/clustering/` and `figures/kmeans_demo/`, tables to `results/`.

## 2. Imports and Configuration

All randomness is controlled by a single `RANDOM_STATE`, so the notebook produces the same clusters every time it is run.

In [6]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import linkage, dendrogram

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (8, 5), 'figure.dpi': 100,
                     'axes.titlesize': 13, 'axes.labelsize': 11})
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

# ---- Reproducibility settings -------------------------------------------
RANDOM_STATE = 42          # fixed seed for every K-Means run
N_INIT = 20                # number of k-means++ initialisations per model
K_RANGE = range(2, 11)     # candidate numbers of clusters

# ---- Paths (relative to the project folder) --------------------------------
PROJECT_DIR = Path.cwd()
RAW_PATH = PROJECT_DIR / 'Online Retail.csv'
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
FIG_DIR = PROJECT_DIR / 'figures' / 'clustering'
DEMO_FIG_DIR = PROJECT_DIR / 'figures' / 'kmeans_demo'
for d in (DATA_DIR, RESULTS_DIR, FIG_DIR, DEMO_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert RAW_PATH.exists(), f'Raw dataset not found at {RAW_PATH}. Run the notebook from the G3 project folder.'

# Colours used for clusters in every figure (Cluster 1, 2, 3, 4)
CLUSTER_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
RFM_COLS = ['Recency', 'Frequency', 'Monetary']

def save_fig(fig, name, folder=FIG_DIR):
    fig.savefig(folder / name, dpi=200, bbox_inches='tight', facecolor='white')

print('Project folder :', PROJECT_DIR)
print('Random state   :', RANDOM_STATE, '| n_init:', N_INIT, '| candidate K:', list(K_RANGE))

Project folder : C:\Users\Levi\Desktop\Classes\SOEN 4.1\SOEN 400 - Data Mining & Analytics\Project\Gohons
Random state   : 42 | n_init: 20 | candidate K: [2, 3, 4, 5, 6, 7, 8, 9, 10]


## 3. Load the Dataset

The UCI Online Retail dataset (Chen, 2015) is loaded from the CSV copy used in Phase 1. It is **transaction-level**: each row is one product line on an invoice, so one customer can have many rows.

In [7]:
def read_csv_robust(path):
    # Same loader as Phase 1: try common encodings in order.
    for enc in ('utf-8', 'utf-8-sig', 'latin-1', 'cp1252'):
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded with encoding='{enc}'")
            return df
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f'Could not read {path} with any common encoding.')

raw = read_csv_robust(RAW_PATH)
print(f'Raw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
assert raw.shape == (541909, 8), 'Unexpected dataset shape - is this the UCI Online Retail file?'
display(raw.head())

Loaded with encoding='latin-1'
Raw dataset: 541,909 rows x 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01/12/2010 08:26,2.550,"17,850.000",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01/12/2010 08:26,3.390,"17,850.000",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01/12/2010 08:26,2.750,"17,850.000",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01/12/2010 08:26,3.390,"17,850.000",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01/12/2010 08:26,3.390,"17,850.000",United Kingdom


## 4. Data Cleaning (Phase 1 decisions, reproduced)

### 4.1 Data-quality issues in the raw data
These counts reproduce the Phase 1 data-quality analysis. Counts overlap (for example, every cancelled invoice line also has a negative quantity), so they must not be added together.

In [8]:
invoice = raw['InvoiceNo'].astype(str)
quality = pd.DataFrame({
    'Issue': ['Missing CustomerID', 'Missing Description', 'Exact duplicate rows',
              'Cancelled invoices (InvoiceNo starts with C)', 'Quantity <= 0', 'UnitPrice <= 0'],
    'Rows': [raw['CustomerID'].isna().sum(), raw['Description'].isna().sum(), raw.duplicated().sum(),
             invoice.str.startswith('C').sum(), (raw['Quantity'] <= 0).sum(), (raw['UnitPrice'] <= 0).sum()],
})
quality['% of raw rows'] = (quality['Rows'] / len(raw) * 100).round(2)
display(quality)

cancelled = invoice.str.startswith('C')
print('Overlap check:')
print(f"  cancelled lines that also have Quantity <= 0        : {(cancelled & (raw['Quantity'] <= 0)).sum():,} of {cancelled.sum():,}")
print(f"  Quantity <= 0 lines that are NOT cancelled           : {(~cancelled & (raw['Quantity'] <= 0)).sum():,}")
print(f"  ... of which have a CustomerID                       : {(~cancelled & (raw['Quantity'] <= 0) & raw['CustomerID'].notna()).sum():,}")

,Issue,Rows,% of raw rows
0,Missing CustomerID,135080,24.930
1,Missing Description,1454,0.270
2,Exact duplicate rows,5268,0.970
3,Cancelled invoices (InvoiceNo starts with C),9288,1.710
4,Quantity <= 0,10624,1.960
5,UnitPrice <= 0,2517,0.460


Overlap check:
  cancelled lines that also have Quantity <= 0        : 9,288 of 9,288
  Quantity <= 0 lines that are NOT cancelled           : 1,336
  ... of which have a CustomerID                       : 0


### 4.2 Cleaning steps
The Phase 1 rules are applied **in the same order as Phase 1**. The reasons are preserved from the Phase 1 report:

| Rule | Reason |
|---|---|
| Drop rows with missing `CustomerID` | The transaction cannot be reliably assigned to an individual customer, and the unit of analysis is the customer. |
| Drop cancelled invoices (`InvoiceNo` starts with `C`) | The initial RFM representation describes positive purchasing behaviour. |
| Keep `Quantity > 0` | Negative quantities can be legitimate reversal/return-type events; they are excluded **because** this RFM model characterises positive purchasing, not because they are "bad data". |
| Keep `UnitPrice > 0` | Needed for a positive sales value. The dataset does not say *why* some prices are zero, so no reason is assumed. |
| Drop exact duplicate rows | Prevents artificially inflated transaction counts and spending. |

The table below shows how many rows each rule removes **at its position in the sequence**. Because the issues overlap, these numbers differ from the raw counts in 4.1.

In [9]:
steps = [('Raw dataset', len(raw))]
clean = raw.copy()
clean = clean.dropna(subset=['CustomerID']);                          steps.append(('Drop missing CustomerID', len(clean)))
clean = clean[~clean['InvoiceNo'].astype(str).str.startswith('C')];   steps.append(('Drop cancelled invoices', len(clean)))
clean = clean[clean['Quantity'] > 0];                                  steps.append(('Keep Quantity > 0', len(clean)))
clean = clean[clean['UnitPrice'] > 0];                                 steps.append(('Keep UnitPrice > 0', len(clean)))
clean = clean.drop_duplicates();                                       steps.append(('Drop exact duplicates', len(clean)))

cleaning_log = pd.DataFrame(steps, columns=['Step', 'Rows remaining'])
cleaning_log['Rows removed'] = (-cleaning_log['Rows remaining'].diff()).fillna(0).astype(int)
cleaning_log['% of raw retained'] = (cleaning_log['Rows remaining'] / len(raw) * 100).round(2)
display(cleaning_log)
cleaning_log.to_csv(RESULTS_DIR / 'cleaning_steps.csv', index=False)

retention = len(clean) / len(raw) * 100
print(f'Rows retained: {len(clean):,} of {len(raw):,} = {retention:.2f}%')
assert len(clean) == 392692 and round(retention, 2) == 72.46

,Step,Rows remaining,Rows removed,% of raw retained
0,Raw dataset,541909,0,100.000
1,Drop missing CustomerID,406829,135080,75.070
2,Drop cancelled invoices,397924,8905,73.430
3,Keep Quantity > 0,397924,0,73.430
4,Keep UnitPrice > 0,397884,40,73.420
5,Drop exact duplicates,392692,5192,72.460


Rows retained: 392,692 of 541,909 = 72.46%


> **Correction carried forward.** 392,692 / 541,909 = **72.46 %** of the original records are retained. A figure of "2.46 %" that circulated in an earlier draft is a typographical error; 72.46 % is the value computed above.

## 5. Build the Customer-Level RFM Table

The transaction rows are aggregated to **one row per customer** (same definitions as Phase 1):

| Feature | Definition | Direction |
|---|---|---|
| **Recency** | Days between the customer's most recent purchase and the snapshot date (last invoice date in the data + 1 day) | Lower = purchased more recently |
| **Frequency** | Number of distinct invoices | Higher = more repeat purchasing |
| **Monetary** | Sum of `Quantity x UnitPrice` over the retained transactions (£) | Higher = more recorded purchasing value |

*Monetary is gross positive purchasing value*, not net spending after returns or cancellations, because those records were excluded above.

The date format (`MM/DD/YYYY HH:MM`) is now given explicitly; Phase 1 inferred it. The result is identical, which the check against the saved Phase 1 file confirms.

In [10]:
clean['InvoiceDate'] = pd.to_datetime(clean['InvoiceDate'], format='%m/%d/%Y %H:%M:%S')
clean['TotalPrice'] = clean['Quantity'] * clean['UnitPrice']
clean['CustomerID'] = clean['CustomerID'].astype(int)

snapshot_date = clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Transactions from {clean['InvoiceDate'].min().date()} to {clean['InvoiceDate'].max().date()}; snapshot date = {snapshot_date.date()}")

rfm = clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda s: (snapshot_date - s.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum'),
).reset_index()

print(f'RFM table: {len(rfm):,} customers x {len(RFM_COLS)} features')
print(f"Distinct CustomerIDs in raw data: {raw['CustomerID'].nunique():,} "
      f"-> {raw['CustomerID'].nunique() - len(rfm)} customers had no retained positive purchase")
assert rfm['CustomerID'].is_unique and len(rfm) == 4338

# Verify against the RFM table saved in Phase 1 (no accidental change or duplication)
saved_path = DATA_DIR / 'rfm_customers.csv'
if saved_path.exists():
    saved = pd.read_csv(saved_path)
    merged = saved.merge(rfm, on='CustomerID', suffixes=('_phase1', '_now'), how='outer', indicator=True)
    assert (merged['_merge'] == 'both').all(), 'Customer sets differ from Phase 1'
    for col in RFM_COLS:
        assert np.allclose(merged[f'{col}_phase1'], merged[f'{col}_now']), f'{col} differs from Phase 1'
    print('Verified: identical to the Phase 1 file data/rfm_customers.csv')

rfm.to_csv(saved_path, index=False)
display(rfm.head())

ValueError: time data "01/12/2010 08:26" doesn't match format "%m/%d/%Y %H:%M:%S". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

## 6. RFM Exploration (summary of Phase 1 findings)

The full EDA is in the Phase 1 notebook. Here we only recompute the statistics that drive the clustering decisions: skewness, correlation and scale.

In [ ]:
desc = rfm[RFM_COLS].describe().T
desc['skewness'] = rfm[RFM_COLS].skew()
display(desc.round(2))

print('Pearson correlations:')
display(rfm[RFM_COLS].corr().round(3))

In [ ]:
scale = pd.DataFrame({
    'min': rfm[RFM_COLS].min(), 'max': rfm[RFM_COLS].max(),
    'range': rfm[RFM_COLS].max() - rfm[RFM_COLS].min(),
    'variance': rfm[RFM_COLS].var(),
})
scale['share of total variance (%)'] = scale['variance'] / scale['variance'].sum() * 100
display(scale.round(4))

**What this shows.**
* Frequency (skew ≈ 12.07) and Monetary (skew ≈ 19.34) are **highly right-skewed**: most customers buy rarely and spend modestly, while a few buy very often or spend very large amounts. Recency is moderately skewed (≈ 1.25).
* Frequency and Monetary have a **moderate positive** correlation (r ≈ 0.553); Recency is weakly and negatively related to both.
* **Scale.** Monetary's range (≈ £280,202) dwarfs Recency (373 days) and Frequency (208 invoices). Monetary accounts for more than 99.9 % of the total variance of the raw features, so a Euclidean distance on the raw values would be decided almost entirely by Monetary.

## 7. Feature Transformation (log1p)

**Why.** K-Means assigns customers by Euclidean distance to the centroids and places each centroid at the *mean* of its members. Means and squared distances are pulled strongly by extreme values. With skew of 12 and 19, a few very large customers would dominate both.

**What.** `log1p(x) = ln(1 + x)` is applied to **Frequency** and **Monetary** (all values are ≥ 1 and > 0, and `log1p` is safe at small values). It compresses the long right tail while keeping the order of customers unchanged: a customer who spends more still has a larger transformed value.

**Recency is not log-transformed** (Phase 1 plan: transform the *strongly* skewed variables). Its skew of 1.25 is moderate; section 16 tests log-transforming it as a sensitivity check.

**Outliers are kept.** Phase 1 decided not to delete extreme customers because very high spending or frequency can be genuine behaviour, and such customers are exactly what segmentation should find. Transformation and scaling reduce their *influence* on the distances without removing them. Section 16 tests removing them as a sensitivity check only.

In [ ]:
rfm_log = rfm[RFM_COLS].copy()
rfm_log['Frequency'] = np.log1p(rfm_log['Frequency'])
rfm_log['Monetary'] = np.log1p(rfm_log['Monetary'])

skew_table = pd.DataFrame({'skew before': rfm[RFM_COLS].skew(), 'skew after': rfm_log.skew()})
skew_table['transformed?'] = ['no', 'log1p', 'log1p']
display(skew_table.round(2))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for j, col in enumerate(RFM_COLS):
    axes[0, j].hist(rfm[col], bins=50, color='#2a78d6', edgecolor='white')
    axes[0, j].set_title(f'{col} – original (skew {rfm[col].skew():.2f})')
    axes[1, j].hist(rfm_log[col], bins=50, color='#1baf7a', edgecolor='white')
    label = 'unchanged' if col == 'Recency' else 'log1p'
    axes[1, j].set_title(f'{col} – {label} (skew {rfm_log[col].skew():.2f})')
    axes[1, j].set_xlabel(col if col == 'Recency' else f'log(1 + {col})')
for ax in axes[:, 0]:
    ax.set_ylabel('Customers')
fig.suptitle('Effect of the log1p transformation on the RFM features', fontsize=14)
fig.tight_layout()
save_fig(fig, '10_log_transform_before_after.png')
plt.show()

## 8. Feature Scaling (standardization)

**Why.** Even after the log transform, the features are on different scales (Recency is 1–374 days; log Monetary is about 1.6–12.5). K-Means has no notion of units — it simply adds up squared differences — so the feature with the largest spread would still dominate.

**What.** `StandardScaler` converts each feature to a z-score: `z = (x − mean) / standard deviation`. After scaling every feature has mean 0 and standard deviation 1, so each contributes on a comparable scale to the distance.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)                   # clustering matrix used from here on
X_df = pd.DataFrame(X, columns=[f'{c}_z' for c in RFM_COLS])

check = pd.DataFrame({'mean': X_df.mean(), 'std': X_df.std(ddof=0)})
display(check.round(4))

variance_share = pd.DataFrame({
    'raw features (%)': rfm[RFM_COLS].var() / rfm[RFM_COLS].var().sum() * 100,
    'after log1p + scaling (%)': pd.Series(X.var(axis=0), index=RFM_COLS) / X.var(axis=0).sum() * 100,
})
print('Share of total variance contributed by each feature:')
display(variance_share.round(4))

## 9. How K-Means Works — a Small Worked Example

Before applying K-Means to 4,338 customers, the algorithm is traced by hand on **nine 2-D points** so that every step is visible. (These points are illustrative only; they are not retail data.)

**K-Means (Lloyd's algorithm)**
1. **Choose K** – the number of clusters.
2. **Initialise** – pick K starting centroids.
3. **Assignment step** – compute the distance from every point to every centroid and assign each point to the *nearest* centroid.
4. **Update step** – move each centroid to the *mean* of the points assigned to it.
5. **Iterate** – repeat steps 3–4.
6. **Converge / stop** – when no point changes cluster (equivalently, the centroids stop moving), or a maximum number of iterations is reached.

The objective K-Means minimises is the **within-cluster sum of squares (WCSS, "inertia")**: $\sum_{k}\sum_{x \in C_k} \lVert x - \mu_k \rVert^2$. Each assignment step and each update step can only lower (or keep) this value, which is why the algorithm converges.

**Euclidean distance** (used by standard K-Means): $d(x, c) = \sqrt{\sum_i (x_i - c_i)^2}$

To show that a bad start is corrected, the two initial centroids are deliberately placed in the *same* group (points A and C).

In [ ]:
toy = pd.DataFrame({'x1': [1, 2, 2, 3, 6, 7, 7, 8, 5],
                    'x2': [2, 1, 3, 2, 6, 5, 7, 6, 4]},
                   index=list('ABCDEFGHI')).astype(float)
P = toy.to_numpy()

def euclidean(a, b):
    return np.sqrt(((a - b) ** 2).sum(axis=-1))

def manhattan(a, b):
    return np.abs(a - b).sum(axis=-1)

centroids = toy.loc[['A', 'C']].to_numpy().copy()     # deliberately poor initial centroids
history, demo_rows = [], []
for iteration in range(1, 11):
    dist = np.column_stack([euclidean(P, c) for c in centroids])      # assignment step
    labels = dist.argmin(axis=1)
    new_centroids = np.array([P[labels == k].mean(axis=0) for k in range(len(centroids))])  # update step
    history.append((centroids.copy(), labels.copy(), new_centroids.copy()))
    for name, (d1, d2), lab in zip(toy.index, dist, labels):
        demo_rows.append({'iteration': iteration, 'point': name, 'x1': toy.loc[name, 'x1'], 'x2': toy.loc[name, 'x2'],
                          'dist_to_c1': round(d1, 3), 'dist_to_c2': round(d2, 3), 'assigned_to': f'c{lab + 1}'})
    wcss = sum(((P[labels == k] - new_centroids[k]) ** 2).sum() for k in range(len(centroids)))
    print(f'Iteration {iteration}: c1 = {np.round(centroids[0], 3)}, c2 = {np.round(centroids[1], 3)} '
          f'-> clusters {{c1: {list(toy.index[labels == 0])}, c2: {list(toy.index[labels == 1])}}} '
          f'-> new c1 = {np.round(new_centroids[0], 3)}, new c2 = {np.round(new_centroids[1], 3)}, WCSS = {wcss:.2f}')
    if np.allclose(new_centroids, centroids):
        print(f'Converged at iteration {iteration}: no point changed cluster and the centroids did not move.')
        break
    centroids = new_centroids

demo_table = pd.DataFrame(demo_rows)
demo_table.to_csv(RESULTS_DIR / 'kmeans_demo_iterations.csv', index=False)
print('\nIteration 1 distance table (assignment step):')
display(demo_table[demo_table['iteration'] == 1].set_index('point'))

In [ ]:
n_panels = len(history)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4.6), sharex=True, sharey=True)
for ax, (i, (c_old, lab, c_new)) in zip(axes, enumerate(history, start=1)):
    for k in range(2):
        ax.scatter(P[lab == k, 0], P[lab == k, 1], s=120, color=CLUSTER_COLORS[k],
                   edgecolor='white', linewidth=1.5, label=f'assigned to c{k + 1}', zorder=3)
        ax.scatter(*c_old[k], marker='X', s=220, color=CLUSTER_COLORS[k], edgecolor='black', zorder=4)
        if not np.allclose(c_old[k], c_new[k]):
            ax.annotate('', xy=c_new[k], xytext=c_old[k],
                        arrowprops=dict(arrowstyle='->', color='black', lw=1.5, ls='--'), zorder=5)
            ax.scatter(*c_new[k], marker='X', s=220, facecolor='white', edgecolor=CLUSTER_COLORS[k], lw=2, zorder=4)
    for name, (x, y) in zip(toy.index, P):
        ax.text(x + 0.18, y + 0.18, name, fontsize=10)
    status = 'converged – nothing changes' if np.allclose(c_old, c_new) else 'centroids move to cluster means'
    ax.set_title(f'Iteration {i}: {status}', fontsize=11)
    ax.set_xlabel('x1')
    ax.set_xlim(0, 9); ax.set_ylim(0, 8)
axes[0].set_ylabel('x2')
axes[0].legend(loc='upper left', fontsize=9)
fig.suptitle('K-Means on 9 points (K = 2). X = centroid used for assignment; hollow X = updated centroid', fontsize=12)
fig.tight_layout()
save_fig(fig, 'kmeans_toy_iterations.png', DEMO_FIG_DIR)
plt.show()

**Reading the example.** In iteration 1 both centroids start inside the left-hand group, so centroid c2 captures C, D and all the right-hand points. The update step moves c2 to the mean of those points, towards the right-hand group. In iteration 2, C and D are now closer to c1 and switch cluster. In iteration 3 no point changes cluster, so the algorithm has converged. The final clusters are {A, B, C, D} and {E, F, G, H, I}.

### 9.1 Euclidean vs Manhattan distance
* **Euclidean** – straight-line distance: $\sqrt{\sum_i (x_i-c_i)^2}$.
* **Manhattan (city-block)** – sum of absolute differences along each axis: $\sum_i |x_i-c_i|$.

Standard K-Means uses (squared) **Euclidean** distance: the mean is exactly the point that minimises the sum of squared Euclidean distances, which is why the update step uses the mean. Manhattan distance is naturally paired with the *median* (the K-Medians / K-Medoids family) and is less affected by a single large difference. This project uses standard Euclidean K-Means.

In [ ]:
a, b = toy.loc['A'].to_numpy(), toy.loc['H'].to_numpy()
print(f'Point A = {a}, point H = {b}')
print(f'Euclidean: sqrt(({b[0]:.0f}-{a[0]:.0f})^2 + ({b[1]:.0f}-{a[1]:.0f})^2) = sqrt({(b[0]-a[0])**2:.0f} + {(b[1]-a[1])**2:.0f}) = {euclidean(a, b):.3f}')
print(f'Manhattan: |{b[0]:.0f}-{a[0]:.0f}| + |{b[1]:.0f}-{a[1]:.0f}| = {abs(b[0]-a[0]):.0f} + {abs(b[1]-a[1]):.0f} = {manhattan(a, b):.0f}')

# The same idea on two real customers (standardized log-RFM space)
i, j = rfm.index[rfm['CustomerID'] == 12347][0], rfm.index[rfm['CustomerID'] == 12348][0]
print('\nTwo real customers:')
display(rfm.loc[[i, j]])
print(f'Euclidean distance on RAW RFM      : {euclidean(rfm.loc[i, RFM_COLS].to_numpy(float), rfm.loc[j, RFM_COLS].to_numpy(float)):,.2f}  (dominated by Monetary)')
print(f'Euclidean distance on scaled log-RFM: {euclidean(X[i], X[j]):.3f}')
print(f'Manhattan distance on scaled log-RFM: {manhattan(X[i], X[j]):.3f}')

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.scatter(*a, s=120, color=CLUSTER_COLORS[0], zorder=3); ax.scatter(*b, s=120, color=CLUSTER_COLORS[1], zorder=3)
ax.plot([a[0], b[0]], [a[1], b[1]], color='black', lw=2, label=f'Euclidean = {euclidean(a, b):.2f}')
ax.plot([a[0], b[0], b[0]], [a[1], a[1], b[1]], color='#6B7280', lw=2, ls='--', label=f'Manhattan = {manhattan(a, b):.0f}')
ax.text(a[0] - 0.5, a[1] - 0.5, 'A (1, 2)'); ax.text(b[0] - 0.5, b[1] + 0.3, 'H (8, 6)')
ax.set_xlim(0, 9); ax.set_ylim(0, 8); ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Euclidean vs Manhattan distance'); ax.legend(loc='upper left')
fig.tight_layout(); save_fig(fig, 'distance_euclidean_vs_manhattan.png', DEMO_FIG_DIR); plt.show()

### 9.2 The same algorithm written out, applied to the real customers
The function below is a plain NumPy version of the loop above. It is run on the standardized RFM matrix and compared with scikit-learn's `KMeans` in section 12, to show that the library performs exactly these steps (scikit-learn adds the smarter *k-means++* initialisation and several restarts).

In [ ]:
def kmeans_from_scratch(data, k, seed, max_iter=300):
    rng = np.random.default_rng(seed)
    centroids = data[rng.choice(len(data), size=k, replace=False)]           # 1. random initial centroids
    for it in range(1, max_iter + 1):
        dist = np.linalg.norm(data[:, None, :] - centroids[None, :, :], axis=2)  # 2. distances to centroids
        labels = dist.argmin(axis=1)                                            #    assign to nearest
        new_centroids = np.array([data[labels == c].mean(axis=0) for c in range(k)])  # 3. update = mean
        if np.allclose(new_centroids, centroids):                              # 4. converged?
            break
        centroids = new_centroids
    inertia = ((data - centroids[labels]) ** 2).sum()
    return labels, centroids, inertia, it

### 9.3 Contrast: hierarchical clustering (dendrogram of the same nine points)
Hierarchical clustering does not need K in advance. **Agglomerative** (bottom-up) clustering starts with every point as its own cluster and repeatedly merges the two closest clusters; **divisive** (top-down) clustering starts with one cluster and repeatedly splits it. The merge history is drawn as a **dendrogram**; cutting it at a chosen height gives a set of clusters. It is used here only to illustrate the concept; the project's production model is K-Means.

In [ ]:
from scipy.cluster.hierarchy import set_link_color_palette
set_link_color_palette(CLUSTER_COLORS[:2])
Z_link = linkage(P, method='ward')   # agglomerative, Ward linkage (merges that least increase WCSS)
fig, ax = plt.subplots(figsize=(7, 4.2))
dendrogram(Z_link, labels=list(toy.index), ax=ax, color_threshold=6, above_threshold_color='#6B7280')
ax.axhline(6, color='#e34948', ls='--', lw=1.5)
ax.text(ax.get_xlim()[1] * 0.99, 6.3, 'cut here -> 2 clusters', ha='right', color='#e34948')
ax.set_title('Agglomerative clustering of the 9 toy points (Ward linkage)')
ax.set_ylabel('Merge distance')
fig.tight_layout(); save_fig(fig, 'dendrogram_toy.png', DEMO_FIG_DIR); plt.show()

## 10. Selecting K — Candidate Models

The data contain no predefined customer groups, so K must be chosen from evidence. K-Means is fitted for every K from 2 to 10 (each with `n_init = 20` k-means++ starts and `random_state = 42`), and three internal measures are recorded:

| Measure | What it measures | Better when |
|---|---|---|
| **Inertia (WCSS)** | Total squared distance of customers to their own centroid (cohesion). Always falls as K grows. | Look for the "elbow" where extra clusters stop helping much |
| **Silhouette score** | For each customer: *(b − a) / max(a, b)*, where *a* = mean distance to its own cluster (cohesion) and *b* = mean distance to the nearest other cluster (separation). Averaged over customers; range −1 to 1. | Higher |
| **Calinski-Harabasz** | Ratio of between-cluster dispersion to within-cluster dispersion, adjusted for K and n. | Higher |

In [ ]:
rows = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=N_INIT, random_state=RANDOM_STATE).fit(X)
    rows.append({'K': k, 'inertia': km.inertia_,
                 'silhouette': silhouette_score(X, km.labels_),
                 'calinski_harabasz': calinski_harabasz_score(X, km.labels_),
                 'smallest cluster': np.bincount(km.labels_).min(),
                 'cluster sizes': sorted(np.bincount(km.labels_).tolist(), reverse=True)})
k_metrics = pd.DataFrame(rows)
k_metrics['inertia drop vs K-1 (%)'] = (-k_metrics['inertia'].pct_change() * 100).round(1)
display(k_metrics)
k_metrics.drop(columns='cluster sizes').to_csv(RESULTS_DIR / 'k_selection_metrics.csv', index=False)

### 10.1 Elbow method

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(k_metrics['K'], k_metrics['inertia'], marker='o', lw=2, ms=8, color='#2a78d6')
for _, r in k_metrics.iterrows():
    if not np.isnan(r['inertia drop vs K-1 (%)']):
        ax.annotate(f"-{r['inertia drop vs K-1 (%)']:.1f}%", (r['K'], r['inertia']),
                    textcoords='offset points', xytext=(9, 7), ha='left', fontsize=9, color='#52514e')
ax.axvline(3, color='#9CA3AF', ls=':', lw=1.2, zorder=0)
ax.set_xticks(list(K_RANGE))
ax.set_xlabel('Number of clusters K'); ax.set_ylabel('Inertia (within-cluster sum of squares)')
ax.set_title('Elbow method: inertia by K (labels = % drop from K-1)')
fig.tight_layout(); save_fig(fig, '11_elbow_method.png'); plt.show()

**Reading the elbow.** Inertia always falls as K increases, so the question is where the *improvement* flattens. Moving from 2 to 3 clusters reduces inertia by 37.5 % and 3 → 4 by 24.6 %; after K = 4 each extra cluster reduces it by 14.9 % or less. The bend is therefore in the **K = 3–4** region. The elbow on its own does not identify a single K, so silhouette and Calinski-Harabasz are used as well.

## 11. Silhouette Analysis (and Calinski-Harabasz)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].plot(k_metrics['K'], k_metrics['silhouette'], marker='o', lw=2, ms=8, color='#2a78d6')
for _, r in k_metrics.iterrows():
    axes[0].annotate(f"{r['silhouette']:.3f}", (r['K'], r['silhouette']), textcoords='offset points',
                     xytext=(7, 6), ha='left', fontsize=9)
axes[0].set_title('Mean silhouette score by K (higher = better)')
axes[0].set_ylabel('Mean silhouette score')
axes[1].plot(k_metrics['K'], k_metrics['calinski_harabasz'], marker='o', lw=2, ms=8, color='#eb6834')
for _, r in k_metrics.iterrows():
    axes[1].annotate(f"{r['calinski_harabasz']:,.0f}", (r['K'], r['calinski_harabasz']), textcoords='offset points',
                     xytext=(7, 6), ha='left', fontsize=9)
axes[1].set_title('Calinski-Harabasz index by K (higher = better)')
axes[1].set_ylabel('Calinski-Harabasz index')
for ax in axes:
    ax.set_xticks(list(K_RANGE)); ax.set_xlabel('Number of clusters K'); ax.axvline(3, color='#9CA3AF', ls=':', lw=1.2, zorder=0)
    ax.margins(x=0.08, y=0.15)
fig.tight_layout(); save_fig(fig, '12_silhouette_and_ch_by_k.png'); plt.show()

best_sil = int(k_metrics.loc[k_metrics['silhouette'].idxmax(), 'K'])
best_ch = int(k_metrics.loc[k_metrics['calinski_harabasz'].idxmax(), 'K'])
print(f'Highest mean silhouette: K = {best_sil}')
print(f'Highest Calinski-Harabasz: K = {best_ch}')
display(k_metrics.set_index('K').loc[[2, 3, 4, 5], ['silhouette', 'calinski_harabasz', 'inertia drop vs K-1 (%)', 'cluster sizes']])

**Reading silhouette and Calinski-Harabasz.**
* The mean silhouette is highest at **K = 3 (≈ 0.416)**, closely followed by K = 2 (≈ 0.406) and K = 4 (≈ 0.380), and falls steadily after that. Values around 0.4 indicate clusters that are reasonably but not perfectly separated — customer behaviour forms a continuum rather than completely distinct islands.
* Calinski-Harabasz is also highest at **K = 3**, with K = 4 very close.

## 12. Final K-Means Model

### 12.1 Choice of K = 3
| Evidence | Points to |
|---|---|
| Elbow | bend in the K = 3–4 region |
| Silhouette | K = 3 highest (K = 2 and K = 4 close behind) |
| Calinski-Harabasz | K = 3 highest (K = 4 close behind) |
| Interpretability | K = 2 only splits "active" from "inactive". K = 3 separates three behaviourally distinct groups, each large enough to act on (section 14). K = 4 separates a smaller top-value group and a middle tier; it is kept as a documented alternative in section 16. |

**K = 3 is selected** because it is the only value supported by all three measures and it gives clearly interpretable groups. This is a judgement based on evidence; it does not mean that three is the "true" number of customer types.

### 12.2 Model settings
* `init='k-means++'` – spreads the initial centroids apart (a better start than purely random centroids).
* `n_init=20` – runs K-Means 20 times from different starts and keeps the run with the lowest inertia, protecting against a poor local optimum.
* `max_iter=300`, `tol=1e-4` – stopping rule: stop when centroids move less than the tolerance, or after 300 iterations.
* `random_state=42` – reproducible results.

In [ ]:
FINAL_K = 3

kmeans = KMeans(n_clusters=FINAL_K, init='k-means++', n_init=N_INIT, max_iter=300, tol=1e-4,
                random_state=RANDOM_STATE)
raw_labels = kmeans.fit_predict(X)

print(f'K = {FINAL_K}')
print(f'Iterations until convergence (best of {N_INIT} runs): {kmeans.n_iter_}')
print(f'Final inertia (WCSS): {kmeans.inertia_:,.2f}')

# Our NumPy implementation from section 9.2 reaches the same partition
scratch_labels, scratch_centroids, scratch_inertia, scratch_iters = kmeans_from_scratch(X, FINAL_K, seed=RANDOM_STATE)
print(f'\nFrom-scratch K-Means: converged after {scratch_iters} iterations, inertia = {scratch_inertia:,.2f}')
print(f'Agreement with scikit-learn (Adjusted Rand Index, 1 = identical partition): '
      f'{adjusted_rand_score(raw_labels, scratch_labels):.4f}')

The NumPy loop from section 9.2 lands on essentially the same partition as scikit-learn (ARI close to 1). The few customers that differ lie on cluster boundaries: scikit-learn stops once the centroids move less than `tol`, while the plain loop runs until the centroids stop moving entirely.

## 13. Cluster Assignment

K-Means numbers clusters arbitrarily (0, 1, 2 can change with the seed). To make the labels stable and easy to talk about, clusters are renumbered **1, 2, 3 in descending order of mean Monetary value**. This renaming does not change which customers are grouped together.

In [ ]:
order = (pd.Series(rfm['Monetary'].values).groupby(raw_labels).mean()
         .sort_values(ascending=False).index.tolist())
relabel = {old: new for new, old in enumerate(order, start=1)}
rfm['Cluster'] = pd.Series(raw_labels).map(relabel).values
centroids_z = kmeans.cluster_centers_[order]          # centroids in the same (renumbered) order

clustered = rfm.copy()
clustered['log_Frequency'] = rfm_log['Frequency'].values
clustered['log_Monetary'] = rfm_log['Monetary'].values
for c in RFM_COLS:
    clustered[f'{c}_scaled'] = X_df[f'{c}_z'].values
clustered.to_csv(DATA_DIR / 'rfm_customers_clustered.csv', index=False)

print('Customers per cluster:')
display(rfm['Cluster'].value_counts().sort_index().rename('customers').to_frame())
display(clustered.head())

## 14. Cluster Profiling

The profile is computed on the **original, interpretable RFM units** (days, invoices, £), not on the transformed values. Medians are reported alongside means because Frequency and Monetary remain skewed *within* clusters, so a few large customers can pull a cluster mean well above its typical member.

In [ ]:
g = rfm.groupby('Cluster')
profile = pd.DataFrame({
    'Customers': g.size(),
    '% of customers': g.size() / len(rfm) * 100,
    'Recency mean': g['Recency'].mean(), 'Recency median': g['Recency'].median(),
    'Recency min': g['Recency'].min(), 'Recency max': g['Recency'].max(),
    'Frequency mean': g['Frequency'].mean(), 'Frequency median': g['Frequency'].median(),
    'Frequency min': g['Frequency'].min(), 'Frequency max': g['Frequency'].max(),
    'Monetary mean': g['Monetary'].mean(), 'Monetary median': g['Monetary'].median(),
    'Monetary min': g['Monetary'].min(), 'Monetary max': g['Monetary'].max(),
    'Total Monetary (£)': g['Monetary'].sum(),
})
profile['% of total Monetary'] = profile['Total Monetary (£)'] / rfm['Monetary'].sum() * 100
profile['% single-invoice customers'] = g['Frequency'].apply(lambda s: (s == 1).mean() * 100)

overall = pd.DataFrame({
    'Customers': [len(rfm)], '% of customers': [100.0],
    'Recency mean': [rfm['Recency'].mean()], 'Recency median': [rfm['Recency'].median()],
    'Recency min': [rfm['Recency'].min()], 'Recency max': [rfm['Recency'].max()],
    'Frequency mean': [rfm['Frequency'].mean()], 'Frequency median': [rfm['Frequency'].median()],
    'Frequency min': [rfm['Frequency'].min()], 'Frequency max': [rfm['Frequency'].max()],
    'Monetary mean': [rfm['Monetary'].mean()], 'Monetary median': [rfm['Monetary'].median()],
    'Monetary min': [rfm['Monetary'].min()], 'Monetary max': [rfm['Monetary'].max()],
    'Total Monetary (£)': [rfm['Monetary'].sum()], '% of total Monetary': [100.0],
    '% single-invoice customers': [(rfm['Frequency'] == 1).mean() * 100],
}, index=pd.Index(['All customers'], name='Cluster'))
profile_out = pd.concat([profile, overall])
profile_out.round(2).to_csv(RESULTS_DIR / 'cluster_profile.csv')

main_cols = ['Customers', '% of customers', 'Recency mean', 'Recency median', 'Frequency mean',
             'Frequency median', 'Monetary mean', 'Monetary median', '% of total Monetary',
             '% single-invoice customers']
display(profile_out[main_cols].round(1))
print('Ranges within each cluster:')
display(profile_out[['Recency min', 'Recency max', 'Frequency min', 'Frequency max',
                     'Monetary min', 'Monetary max']].round(2))

### 14.1 Relative to the whole customer base
Each cluster median divided by the overall median (1.0 = typical customer; for Recency, **below 1 means more recent**).

In [ ]:
relative = pd.DataFrame({c: profile[f'{c} median'] / rfm[c].median() for c in RFM_COLS})
display(relative.round(2))

# Centroids translated back into original units: undo scaling, then undo log1p
centroid_log = scaler.inverse_transform(centroids_z)
centroids_units = pd.DataFrame(centroid_log, columns=RFM_COLS, index=pd.Index(range(1, FINAL_K + 1), name='Cluster'))
centroids_units['Frequency'] = np.expm1(centroids_units['Frequency'])
centroids_units['Monetary'] = np.expm1(centroids_units['Monetary'])
centroids_std = pd.DataFrame(centroids_z, columns=[f'{c} (z)' for c in RFM_COLS], index=centroids_units.index)
centroid_table = pd.concat([centroids_std, centroids_units.add_suffix(' (original units)')], axis=1)
print('Cluster centroids (standardized, and converted back to days / invoices / £):')
display(centroid_table.round(2))
centroid_table.round(4).to_csv(RESULTS_DIR / 'cluster_centroids.csv')

*Note:* the centroid in original units is the back-transformed mean of the **log** values, i.e. a geometric-mean-like "typical" value. That is why it sits close to the cluster median and below the arithmetic mean for Frequency and Monetary.

## 15. Cluster Visualization
### 15.1 Cluster sizes

In [ ]:
sizes = rfm['Cluster'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar([f'Cluster {c}' for c in sizes.index], sizes.values,
              color=CLUSTER_COLORS[:FINAL_K], edgecolor='white', width=0.6)
ax.bar_label(bars, labels=[f'{v:,}\n({v / len(rfm) * 100:.1f}%)' for v in sizes.values], padding=3)
ax.set_ylabel('Customers'); ax.set_title('Number of customers in each cluster')
ax.set_ylim(0, sizes.max() * 1.22)
fig.tight_layout(); save_fig(fig, '13_cluster_sizes.png'); plt.show()

### 15.2 RFM comparison across clusters
Boxplots show the full spread inside each cluster. Frequency and Monetary are drawn on a log axis so that the typical customers are visible and not squashed by the largest ones.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
palette = {c: CLUSTER_COLORS[c - 1] for c in range(1, FINAL_K + 1)}
units = {'Recency': 'days since last purchase', 'Frequency': 'distinct invoices (log scale)',
         'Monetary': '£ spent (log scale)'}
for ax, col in zip(axes, RFM_COLS):
    sns.boxplot(data=rfm, x='Cluster', y=col, hue='Cluster', palette=palette, legend=False,
                ax=ax, fliersize=2, width=0.6)
    if col != 'Recency':
        ax.set_yscale('log')
    ax.set_title(col); ax.set_ylabel(units[col]); ax.set_xlabel('Cluster')
fig.suptitle('RFM distribution by cluster (original units)', fontsize=14)
fig.tight_layout(); save_fig(fig, '14_rfm_boxplots_by_cluster.png'); plt.show()

In [ ]:
# Standardized centroid profile: how far each cluster's centre is from the average customer (0)
fig, ax = plt.subplots(figsize=(9, 4.8))
xpos = np.arange(len(RFM_COLS))
width = 0.8 / FINAL_K
for idx, c in enumerate(range(1, FINAL_K + 1)):
    bars = ax.bar(xpos + (idx - (FINAL_K - 1) / 2) * width, centroids_z[idx], width=width * 0.92,
                  color=CLUSTER_COLORS[idx], label=f'Cluster {c}')
    ax.bar_label(bars, labels=[f'{v:+.2f}' for v in centroids_z[idx]], padding=3, fontsize=9)
ax.axhline(0, color='black', lw=1)
ax.set_xticks(xpos); ax.set_xticklabels(['Recency\n(higher = LESS recent)', 'log Frequency', 'log Monetary'])
ax.set_ylabel('Centroid (standard deviations from mean)')
ax.set_title('Cluster centroids in the standardized feature space (0 = average customer)')
ax.legend(loc='upper right'); ax.margins(y=0.15)
fig.tight_layout(); save_fig(fig, '15_centroid_profile.png'); plt.show()

### 15.3 Two-dimensional views
**Principal Component Analysis (PCA)** projects the three standardized features onto the two directions that retain the most variance, purely so the clusters can be *drawn* on a flat page. **PCA is only a visualization here — the clustering itself was done in the full 3-feature space**, so some apparent overlap in 2-D is a result of the projection.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
cent_pca = pca.transform(centroids_z)
ev = pca.explained_variance_ratio_
print(f'Variance retained: PC1 {ev[0] * 100:.1f}%, PC2 {ev[1] * 100:.1f}%, total {ev.sum() * 100:.1f}%')
display(pd.DataFrame(pca.components_, columns=RFM_COLS, index=['PC1 loadings', 'PC2 loadings']).round(3))

fig, ax = plt.subplots(figsize=(8.5, 6))
for idx, c in enumerate(range(1, FINAL_K + 1)):
    m = rfm['Cluster'].values == c
    ax.scatter(X_pca[m, 0], X_pca[m, 1], s=10, alpha=0.45, color=CLUSTER_COLORS[idx],
               label=f'Cluster {c} (n = {m.sum():,})')
for idx, (x, y) in enumerate(cent_pca):
    ax.scatter(x, y, marker='X', s=320, color=CLUSTER_COLORS[idx], edgecolor='black', linewidth=1.5, zorder=5)
ax.set_xlabel(f'PC1 ({ev[0] * 100:.1f}% of variance)'); ax.set_ylabel(f'PC2 ({ev[1] * 100:.1f}% of variance)')
ax.set_title('Customer clusters projected onto two principal components\n(visualization only; X = cluster centroid)')
ax.legend(markerscale=2.5, loc='best')
fig.tight_layout(); save_fig(fig, '16_pca_clusters.png'); plt.show()

In [ ]:
# A 2-D view in real units: Recency vs Monetary (log axis)
fig, ax = plt.subplots(figsize=(8.5, 5.6))
for idx, c in enumerate(range(1, FINAL_K + 1)):
    d = rfm[rfm['Cluster'] == c]
    ax.scatter(d['Recency'], d['Monetary'], s=9, alpha=0.45, color=CLUSTER_COLORS[idx], label=f'Cluster {c}')
ax.set_yscale('log')
ax.set_xlabel('Recency (days since last purchase)'); ax.set_ylabel('Monetary (£, log scale)')
ax.set_title('Customers by Recency and Monetary, coloured by cluster')
ax.legend(markerscale=2.5)
fig.tight_layout(); save_fig(fig, '17_recency_vs_monetary_clusters.png'); plt.show()

## 16. Evaluation

Clustering is **unsupervised**: there are no true customer labels to compare against, so no score can prove the clusters are "correct". Instead the clusters are judged on four questions:

1. **Cohesion** – are customers close to their own cluster centre?
2. **Separation** – are clusters far from one another?
3. **Stability** – do we get the same clusters if the random start or reasonable preprocessing choices change?
4. **Interpretability** – do the clusters describe recognisably different purchasing behaviour (section 17)?

### 16.1 Internal metrics for the final model

In [ ]:
sil_samples = silhouette_samples(X, raw_labels)
rfm['silhouette'] = sil_samples
per_cluster_sil = rfm.groupby('Cluster')['silhouette'].agg(['mean', 'median', 'min'])
per_cluster_sil['% customers with negative silhouette'] = rfm.groupby('Cluster')['silhouette'].apply(lambda s: (s < 0).mean() * 100)

final_sil = silhouette_score(X, raw_labels)
final_ch = calinski_harabasz_score(X, raw_labels)
total_ss = ((X - X.mean(axis=0)) ** 2).sum()
between_share = 1 - kmeans.inertia_ / total_ss
print(f'Mean silhouette score     : {final_sil:.3f}')
print(f'Calinski-Harabasz index   : {final_ch:,.1f}')
print(f'Inertia (WCSS)            : {kmeans.inertia_:,.1f} of total sum of squares {total_ss:,.1f}')
print(f'Variance explained by the clustering (between-cluster SS / total SS): {between_share * 100:.1f}%')
print(f'Customers with a negative silhouette (closer to another cluster): {(sil_samples < 0).mean() * 100:.1f}%')
display(per_cluster_sil.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.6))
y_lower = 10
for idx, c in enumerate(range(1, FINAL_K + 1)):
    vals = np.sort(rfm.loc[rfm['Cluster'] == c, 'silhouette'].values)
    ax.fill_betweenx(np.arange(y_lower, y_lower + len(vals)), 0, vals, color=CLUSTER_COLORS[idx], alpha=0.85)
    ax.text(-0.08, y_lower + len(vals) / 2, f'Cluster {c}', ha='right', va='center', fontsize=10)
    y_lower += len(vals) + 40
ax.axvline(final_sil, color='black', ls='--', lw=1.2, label=f'mean silhouette = {final_sil:.3f}')
ax.set_xlabel('Silhouette coefficient'); ax.set_yticks([]); ax.set_xlim(-0.35, 1)
ax.set_title(f'Silhouette plot for the final model (K = {FINAL_K})')
ax.legend(loc='lower right')
fig.tight_layout(); save_fig(fig, '18_silhouette_plot_final.png'); plt.show()

### 16.2 Stability across random starts
K-Means can end in different local optima depending on the initial centroids. The final model is refitted with 10 different seeds and compared with the reference partition using the **Adjusted Rand Index** (ARI: 1 = identical partition, ≈ 0 = no better than chance). The same test is repeated with only one initialisation (`n_init=1`) to show why multiple starts are used.

In [ ]:
def ari_across_seeds(k, n_init, seeds=range(10)):
    ref = KMeans(n_clusters=k, n_init=n_init, random_state=RANDOM_STATE).fit_predict(X)
    scores = [adjusted_rand_score(ref, KMeans(n_clusters=k, n_init=n_init, random_state=s).fit_predict(X))
              for s in seeds if s != RANDOM_STATE]
    inertias = [KMeans(n_clusters=k, n_init=n_init, random_state=s).fit(X).inertia_ for s in seeds]
    return min(scores), np.mean(scores), min(inertias), max(inertias)

stab_rows = []
for k in (FINAL_K, 4, 5):
    for n_init in (1, N_INIT):
        mn, avg, i_min, i_max = ari_across_seeds(k, n_init)
        stab_rows.append({'K': k, 'n_init': n_init, 'min ARI vs reference': mn, 'mean ARI': avg,
                          'inertia range across seeds': f'{i_min:,.1f} to {i_max:,.1f}'})
stability = pd.DataFrame(stab_rows)
display(stability)
stability.to_csv(RESULTS_DIR / 'stability_across_seeds.csv', index=False)

### 16.3 Sensitivity to preprocessing and K
Each variant changes **one** reasonable choice and is compared with the main model using ARI. Silhouette scores computed in *different* feature spaces are not directly comparable, so ARI and cluster sizes are the main things to read here.

| Variant | Change |
|---|---|
| A. Main model | log1p(F, M) + StandardScaler, K = 3 |
| B. No log transform | StandardScaler on raw R, F, M |
| C. Log all three | log1p also applied to Recency |
| D. Min-max scaling | log1p(F, M) + MinMaxScaler instead of StandardScaler |
| E. Top 1 % Monetary removed | Refit without the highest-spending 1 % of customers (outlier-removal check only; the main dataset is **not** changed) |

In [ ]:
def fit_variant(features, k=FINAL_K):
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE).fit(features)
    return km.labels_, silhouette_score(features, km.labels_)

variants = []
main_labels = rfm['Cluster'].values
variants.append(('A. Main model', main_labels, final_sil, len(rfm)))

lab, s = fit_variant(StandardScaler().fit_transform(rfm[RFM_COLS]))
variants.append(('B. No log transform', lab, s, len(rfm)))

lab, s = fit_variant(StandardScaler().fit_transform(np.log1p(rfm[RFM_COLS])))
variants.append(('C. Log all three', lab, s, len(rfm)))

lab, s = fit_variant(MinMaxScaler().fit_transform(rfm_log))
variants.append(('D. Min-max scaling', lab, s, len(rfm)))

keep = (rfm['Monetary'] <= rfm['Monetary'].quantile(0.99)).values
lab_trim, s = fit_variant(StandardScaler().fit_transform(rfm_log[keep]))
variants.append(('E. Top 1% Monetary removed', lab_trim, s, keep.sum()))

sens_rows = []
for name, lab, s, n in variants:
    ref = main_labels if n == len(rfm) else main_labels[keep]
    sens_rows.append({'Variant': name, 'Customers': n, 'Silhouette (own feature space)': round(s, 3),
                      'ARI vs main model': round(adjusted_rand_score(ref, lab), 3),
                      'Cluster sizes (largest first)': sorted(np.unique(lab, return_counts=True)[1].tolist(), reverse=True)})
sensitivity = pd.DataFrame(sens_rows)
display(sensitivity)
sensitivity.to_csv(RESULTS_DIR / 'sensitivity_analysis.csv', index=False)

# What do the two most different variants actually look like?
for name, lab, _, n in variants[1:3]:
    print(f'{name}: median RFM per cluster')
    display(rfm.assign(variant_cluster=lab).groupby('variant_cluster').agg(
        Customers=('Recency', 'size'), Recency_median=('Recency', 'median'),
        Frequency_median=('Frequency', 'median'), Monetary_median=('Monetary', 'median')).round(1))

In [ ]:
# How the K = 3 clusters map onto the K = 4 alternative
labels_k4 = KMeans(n_clusters=4, n_init=N_INIT, random_state=RANDOM_STATE).fit_predict(X)
order4 = pd.Series(rfm['Monetary'].values).groupby(labels_k4).mean().sort_values(ascending=False).index
k4 = pd.Series(labels_k4).map({old: new for new, old in enumerate(order4, start=1)}).values
xtab = pd.crosstab(pd.Series(main_labels, name='K = 3 cluster'), pd.Series(k4, name='K = 4 cluster'))
print('Customers: rows = final K = 3 clusters, columns = K = 4 clusters')
display(xtab)
k4_profile = rfm.assign(K4=k4).groupby('K4').agg(Customers=('Recency', 'size'),
    Recency_median=('Recency', 'median'), Frequency_median=('Frequency', 'median'),
    Monetary_median=('Monetary', 'median'), Monetary_mean=('Monetary', 'mean'))
print('K = 4 alternative profile:')
display(k4_profile.round(1))
k4_profile.round(2).to_csv(RESULTS_DIR / 'alternative_k4_profile.csv')

In [ ]:
eval_summary = pd.DataFrame([
    ('Final K', FINAL_K),
    ('Mean silhouette', round(final_sil, 3)),
    ('Calinski-Harabasz', round(final_ch, 1)),
    ('Inertia (WCSS)', round(kmeans.inertia_, 1)),
    ('Between-cluster SS / total SS (%)', round(between_share * 100, 1)),
    ('Customers with negative silhouette (%)', round((sil_samples < 0).mean() * 100, 1)),
    ('Min ARI across 10 seeds (n_init=20)', round(stability.query('K == @FINAL_K and n_init == @N_INIT')['min ARI vs reference'].iloc[0], 3)),
    ('PCA variance shown in 2-D plot (%)', round(ev.sum() * 100, 1)),
], columns=['Measure', 'Value'])
display(eval_summary)
eval_summary.to_csv(RESULTS_DIR / 'evaluation_summary.csv', index=False)

**What the evaluation shows** (numbers from the cells above).
* **Cohesion and separation.** A mean silhouette of ≈ 0.42 indicates reasonable but not sharp structure — customer behaviour is a continuum, not separate islands. Only about 0.5 % of customers have a negative silhouette (boundary members of Clusters 1 and 3), and the three clusters have similar average silhouettes (≈ 0.41–0.43), so no single cluster is much weaker than the others. The clustering accounts for about 67 % of the total variance of the standardized features (between-cluster / total sum of squares).
* **Stability to the random start.** With 20 initialisations the K = 3 solution is practically identical across 10 seeds (minimum ARI ≈ 0.998). Single-start runs are also stable at K = 3, but at K = 5 a single start can end in a clearly different local optimum (minimum ARI ≈ 0.47, higher inertia) — a concrete example of K-Means' sensitivity to initialisation and the reason `n_init > 1` is used.
* **Robust to the scaler and to extreme spenders.** Min-max scaling instead of standardization (D, ARI ≈ 0.86) and removing the top 1 % of spenders (E, ARI ≈ 0.94) give very similar clusters. The main segments are therefore not an artefact of the scaler or of a few very large customers.
* **Without the log transform (B)** K-Means isolates a tiny group of 26 extreme customers and splits everyone else mainly by Recency. Its silhouette is *higher* (≈ 0.59) because one tiny, far-away group inflates separation — a higher score is not automatically a more useful segmentation. This is the outlier/scale domination the transformation was introduced to prevent.
* **Log-transforming Recency as well (C)** gives a noticeably different partition (ARI ≈ 0.31). On a log scale the difference between 150 and 300 days shrinks, so the lapsed customers merge with part of Cluster 2 into one large low-value group. A top / middle / low value ordering still appears, but **where the boundary between recent and lapsed customers falls depends on how Recency is represented**. The main model keeps Recency in days, as planned in Phase 1: its skew is moderate (1.25) and days since last purchase is directly interpretable. This dependence is reported as a limitation.
* **K = 4 alternative.** The crosstab shows that K = 4 keeps the lapsed group almost unchanged, separates the 571 most valuable customers of Cluster 1 into their own group, and forms a middle tier from the rest of Cluster 1 plus part of Cluster 2. It is a finer-grained version of the same value structure rather than a different pattern.

These scores show the clusters are *reasonable and reproducible*; they do not show they are the only valid segmentation.

## 17. Interpretation

Behaviour is described first, from the numbers; descriptive names are attached only afterwards.

In [ ]:
def describe(c):
    p = profile.loc[c]
    rel = relative.loc[c]
    rec = 'more recent' if rel['Recency'] < 1 else 'less recent'
    return (f"Cluster {c}: {int(p['Customers']):,} customers ({p['% of customers']:.1f}%). "
            f"Median Recency {p['Recency median']:g} days ({rec} than the overall median of {rfm['Recency'].median():.0f}); "
            f"median Frequency {p['Frequency median']:.0f} invoices (overall {rfm['Frequency'].median():.0f}); "
            f"median Monetary £{p['Monetary median']:,.2f} (overall £{rfm['Monetary'].median():,.2f}); "
            f"{p['% single-invoice customers']:.1f}% bought only once; "
            f"together they account for {p['% of total Monetary']:.1f}% of total retained Monetary value.")
for c in profile.index:
    print(describe(c), end='\n\n')

| Cluster | Observed behaviour (from the profile) | Descriptive label |
|---|---|---|
| **1** | Most recent purchases, several invoices per customer, highest spending; a minority of customers accounting for the large majority of retained purchase value | **Active, frequent, high-value customers** |
| **2** | Purchased moderately recently (around the overall median), usually one or two invoices, modest spending | **Recent, low-frequency, lower-value customers** |
| **3** | Last purchase long ago (median about eight months before the snapshot), almost always a single invoice, lowest spending | **Lapsed, mostly one-time, low-value customers** |

**Caveats on interpretation.**
* Labels summarise the *typical* member; the ranges table in section 14 shows each cluster still contains varied customers (e.g. Cluster 1 contains both moderate and extremely large spenders).
* The data cover about one year (1 Dec 2010 – 9 Dec 2011). "Lapsed" means *no retained purchase in this window after an earlier one*; the data cannot say whether these customers have permanently left.
* Monetary is gross positive purchasing value; returns and cancellations were excluded, so net value may be lower for some customers.
* Many customers of this retailer are wholesalers (UCI dataset description), so high-value customers may be business buyers rather than individual consumers; the data do not identify which.

## 18. Conclusions

* The transaction-level data (541,909 rows) were cleaned with the Phase 1 rules, retaining **392,692 rows (72.46 %)**, and aggregated into an RFM table of **4,338 customers**, identical to the Phase 1 output.
* Frequency and Monetary were strongly right-skewed and on very different scales, so they were **log1p-transformed** and all three features were **standardized** before clustering. The sensitivity analysis confirms this step matters: without it K-Means mainly isolates 26 extreme customers.
* K was chosen from evidence. The elbow bends at K = 3–4, and silhouette and Calinski-Harabasz are both highest at **K = 3**, which also gives clearly interpretable groups.
* K-Means (k-means++, 20 initialisations, random_state = 42) produced three reproducible segments: **active high-value customers, recent lower-value customers, and lapsed one-time customers**. The segments are reasonably separated (silhouette ≈ 0.42), very stable across random starts, and robust to the choice of scaler and to removing the top 1 % of spenders; the boundary between recent and lapsed customers does depend on keeping Recency in days rather than log-days.
* Business use: these groups differ enough in recency, frequency and spending to justify different treatment (for example, retaining and rewarding Cluster 1, encouraging repeat purchase in Cluster 2, and deciding whether re-activation of Cluster 3 is worthwhile). The data do **not** test whether such actions would work; that would require a follow-up experiment.

**Outputs:** `data/rfm_customers_clustered.csv`, `results/*.csv`, `figures/clustering/*.png`, `figures/kmeans_demo/*.png`.